# Module 10: Countermeasures — Lab

This lab simulates masking countermeasures and evaluates CPA resistance.

**Objectives:**
1. Implement Boolean masking for AES S-box
2. Compare CPA attacks on masked vs. unmasked implementations
3. Analyze the effect of hiding countermeasures (random delays)
4. Evaluate countermeasure effectiveness using correlation analysis

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

print("=" * 60)
print("BOOLEAN MASKING COUNTERMEASURE")
print("=" * 60)

AES_SBOX = np.array([
    0x63,0x7C,0x77,0x7B,0xF2,0x6B,0x6F,0xC5,0x30,0x01,0x67,0x2B,0xFE,0xD7,0xAB,0x76,
    0xCA,0x82,0xC9,0x7D,0xFA,0x59,0x47,0xF0,0xAD,0xD4,0xA2,0xAF,0x9C,0xA4,0x72,0xC0,
    0xB7,0xFD,0x93,0x26,0x36,0x3F,0xF7,0xCC,0x34,0xA5,0xE5,0xF1,0x71,0xD8,0x31,0x15,
    0x04,0xC7,0x23,0xC3,0x18,0x96,0x05,0x9A,0x07,0x12,0x80,0xE2,0xEB,0x27,0xB2,0x75,
    0x09,0x83,0x2C,0x1A,0x1B,0x6E,0x5A,0xA0,0x52,0x3B,0xD6,0xB3,0x29,0xE3,0x2F,0x84,
    0x53,0xD1,0x00,0xED,0x20,0xFC,0xB1,0x5B,0x6A,0xCB,0xBE,0x39,0x4A,0x4C,0x58,0xCF,
    0xD0,0xEF,0xAA,0xFB,0x43,0x4D,0x33,0x85,0x45,0xF9,0x02,0x7F,0x50,0x3C,0x9F,0xA8,
    0x51,0xA3,0x40,0x8F,0x92,0x9D,0x38,0xF5,0xBC,0xB6,0xDA,0x21,0x10,0xFF,0xF3,0xD2,
    0xCD,0x0C,0x13,0xEC,0x5F,0x97,0x44,0x17,0xC4,0xA7,0x7E,0x3D,0x64,0x5D,0x19,0x73,
    0x60,0x81,0x4F,0xDC,0x22,0x2A,0x90,0x88,0x46,0xEE,0xB8,0x14,0xDE,0x5E,0x0B,0xDB,
    0xE0,0x32,0x3A,0x0A,0x49,0x06,0x24,0x5C,0xC2,0xD3,0xAC,0x62,0x91,0x95,0xE4,0x79,
    0xE7,0xC8,0x37,0x6D,0x8D,0xD5,0x4E,0xA9,0x6C,0x56,0xF4,0xEA,0x65,0x7A,0xAE,0x08,
    0xBA,0x78,0x25,0x2E,0x1C,0xA6,0xB4,0xC6,0xE8,0xDD,0x74,0x1F,0x4B,0xBD,0x8B,0x8A,
    0x70,0x3E,0xB5,0x66,0x48,0x03,0xF6,0x0E,0x61,0x35,0x57,0xB9,0x86,0xC1,0x1D,0x9E,
    0xE1,0xF8,0x98,0x11,0x69,0xD9,0x8E,0x94,0x9B,0x1E,0x87,0xE9,0xCE,0x55,0x28,0xDF,
    0x8C,0xA1,0x89,0x0D,0xBF,0xE6,0x42,0x68,0x41,0x99,0x2D,0x0F,0xB0,0x54,0xBB,0x16,
], dtype=np.uint8)

hw = np.vectorize(lambda b: bin(b).count('1'))

# First-order Boolean masking
def mask_sbox(x, mask):
    """Compute masked S-box: S(x ⊕ m) ⊕ m' """
    masked_input = x ^ mask
    sbox_out = AES_SBOX[masked_input]
    new_mask = np.random.randint(0, 256, dtype=np.uint8)
    masked_output = sbox_out ^ new_mask
    return masked_output, new_mask

def unmask_sbox(masked_output, new_mask):
    """Recover S-box output from masked representation"""
    return masked_output ^ new_mask

# Demonstrate masking
print("\nMasking Example:")
x = 0x42  # Input
m = np.random.randint(0, 256, dtype=np.uint8)  # Random mask
sbox_true = AES_SBOX[x]

masked_out, new_m = mask_sbox(x, m)
sbox_recovered = unmask_sbox(masked_out, new_m)

print(f"  Input x = 0x{x:02X}")
print(f"  True S(x) = 0x{sbox_true:02X}")
print(f"  Mask m = 0x{m:02X}")
print(f"  Masked input = x ⊕ m = 0x{x ^ m:02X}")
print(f"  S(x ⊕ m) = 0x{AES_SBOX[x ^ m]:02X}")
print(f"  New mask m' = 0x{new_m:02X}")
print(f"  Masked output = S(x ⊕ m) ⊕ m' = 0x{masked_out:02X}")
print(f"  Recovered: masked ⊕ m' = 0x{sbox_recovered:02X}")
print(f"  Correct: {sbox_recovered == sbox_true}")

In [ ]:
# Compare CPA attacks on masked vs unmasked
print("=" * 60)
print("CPA RESISTANCE: MASKED vs UNMASKED")
print("=" * 60)

np.random.seed(42)

def pearson(x, y):
    n = len(x)
    mx, my = np.mean(x), np.mean(y)
    dx, dy = x - mx, y - my
    num = np.sum(dx * dy)
    den = np.sqrt(np.sum(dx**2) * np.sum(dy**2))
    return num / den if den > 0 else 0

def run_cpa(traces, plaintexts, key_byte, byte_idx=0, n_samples=100):
    """Run CPA and return max correlation for correct key"""
    best_corr = 0
    best_key = 0
    for k_guess in range(256):
        intermediates = hw(AES_SBOX[plaintexts[:, byte_idx] ^ k_guess]).astype(float)
        for t in range(n_samples):
            corr = abs(pearson(intermediates, traces[:, t]))
            if corr > best_corr:
                best_corr = corr
                best_key = k_guess
    return best_key, best_corr

# Generate traces: unmasked vs masked
secret_key = 0x2B
n_traces_list = [100, 500, 1000, 5000]
n_samples = 100

print(f"{'Traces':>8} {'Unmasked ρ':>12} {'Masked ρ':>12} {'Reduction':>12}")
print("-" * 50)

unmasked_results = []
masked_results = []

for n_tr in n_traces_list:
    # Unmasked traces
    pt = np.random.randint(0, 256, (n_tr, 16), dtype=np.uint8)
    traces_unmasked = np.random.normal(0, 0.3, (n_tr, n_samples))
    for i in range(n_tr):
        sbox_out = AES_SBOX[pt[i, 0] ^ secret_key]
        traces_unmasked[i, 50] += hw(sbox_out) * 0.5
    
    # Masked traces (masking hides the intermediate value)
    traces_masked = np.random.normal(0, 0.3, (n_tr, n_samples))
    for i in range(n_tr):
        mask = np.random.randint(0, 256, dtype=np.uint8)
        masked_input = pt[i, 0] ^ mask
        sbox_out_masked = AES_SBOX[masked_input]  # Intermediate is random!
        new_mask = np.random.randint(0, 256, dtype=np.uint8)
        # Leakage is from S-box output XOR new_mask (random)
        traces_masked[i, 50] += hw(sbox_out_masked ^ new_mask) * 0.5
    
    _, corr_unmasked = run_cpa(traces_unmasked, pt, secret_key)
    _, corr_masked = run_cpa(traces_masked, pt, secret_key)
    
    unmasked_results.append(corr_unmasked)
    masked_results.append(corr_masked)
    reduction = (1 - corr_masked/max(corr_unmasked, 0.001)) * 100
    
    print(f"{n_tr:>8} {corr_unmasked:>12.4f} {corr_masked:>12.4f} {reduction:>11.1f}%")

print("\nMasking reduces correlation → more traces needed for successful CPA")

# Plot comparison
plt.figure(figsize=(10, 5))
plt.plot(n_traces_list, unmasked_results, 'o-', label='Unmasked', linewidth=2)
plt.plot(n_traces_list, masked_results, 's-', label='Masked (1st order)', linewidth=2)
plt.xlabel('Number of Traces')
plt.ylabel('Max Correlation (|ρ|)')
plt.title('CPA Resistance: Masked vs Unmasked AES')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Shuffling countermeasure simulation
print("=" * 60)
print("SHUFFLING COUNTERMEASURE")
print("=" * 60)

np.random.seed(42)

secret_key = np.array([
    0x2B, 0x7E, 0x15, 0x16, 0x28, 0xAE, 0xD2, 0xA6,
    0xAB, 0xF7, 0x15, 0x88, 0x09, 0xCF, 0x4F, 0x3C
], dtype=np.uint8)

n_traces = 500
n_samples = 250

# Unshuffled: each byte at fixed position
pt_unshuffled = np.random.randint(0, 256, (n_traces, 16), dtype=np.uint8)
traces_unshuffled = np.random.normal(0, 0.3, (n_traces, n_samples))
for byte_idx in range(16):
    t_pos = 15 + byte_idx * 14
    for i in range(n_traces):
        sbox_out = AES_SBOX[pt_unshuffled[i, byte_idx] ^ secret_key[byte_idx]]
        traces_unshuffled[i, t_pos] += hw(sbox_out) * 0.5

# Shuffled: each byte at random position
pt_shuffled = np.random.randint(0, 256, (n_traces, 16), dtype=np.uint8)
traces_shuffled = np.random.normal(0, 0.3, (n_traces, n_samples))
for i in range(n_traces):
    perm = np.random.permutation(16)
    for j, byte_idx in enumerate(perm):
        t_pos = 15 + j * 14  # Position depends on permutation
        sbox_out = AES_SBOX[pt_shuffled[i, byte_idx] ^ secret_key[byte_idx]]
        traces_shuffled[i, t_pos] += hw(sbox_out) * 0.5

# CPA on unshuffled (should succeed)
key_unshuffled, corr_unshuffled = run_cpa(traces_unshuffled, pt_unshuffled, secret_key)
rank_unshuffled = np.argsort(-np.array([abs(pearson(hw(AES_SBOX[pt_unshuffled[:, 0] ^ k]).astype(float), traces_unshuffled[:, 50])) for k in range(256)]))[0]

# CPA on shuffled (should fail or require many more traces)
key_shuffled, corr_shuffled = run_cpa(traces_shuffled, pt_shuffled, secret_key)
rank_shuffled = np.argsort(-np.array([abs(pearson(hw(AES_SBOX[pt_shuffled[:, 0] ^ k]).astype(float), traces_shuffled[:, 50])) for k in range(256)]))[0]

print(f"\nUnshuffled CPA:")
print(f"  Recovered key byte 0: 0x{key_unshuffled:02X} (true: 0x{secret_key[0]:02X})")
print(f"  Max correlation: {corr_unshuffled:.4f}")
print(f"  Correct: {'YES' if key_unshuffled == secret_key[0] else 'NO'}")

print(f"\nShuffled CPA (byte 0 at random position):")
print(f"  Recovered key byte 0: 0x{key_shuffled:02X} (true: 0x{secret_key[0]:02X})")
print(f"  Max correlation: {corr_shuffled:.4f}")
print(f"  Correct: {'YES' if key_shuffled == secret_key[0] else 'NO'}")

print("\nShuffling forces attacker to align traces before CPA.")
print("Without correct alignment, CPA correlation is significantly reduced.")

In [ ]:
# Countermeasure evaluation summary
print("=" * 60)
print("COUNTERMEASURE EVALUATION SUMMARY")
print("=" * 60)

countermeasures = [
    {"name": "None", "type": "N/A", "traces": "100-500",
     "overhead": "1×", "security": "None"},
    {"name": "Random Delays", "type": "Hiding", "traces": "500-2000",
     "overhead": "1.5-2×", "security": "Low"},
    {"name": "Shuffling", "type": "Hiding", "traces": "1000-5000",
     "overhead": "1.5-2×", "security": "Low-Medium"},
    {"name": "1st-Order Masking", "type": "Masking", "traces": "10,000-100,000",
     "overhead": "2-4×", "security": "Medium"},
    {"name": "2nd-Order Masking", "type": "Masking", "traces": ">100,000",
     "overhead": "4-8×", "security": "High"},
    {"name": "Threshold Impl.", "type": "Masking", "traces": ">1,000,000",
     "overhead": "3-6×", "security": "Very High"},
    {"name": "Dual-Rail Logic", "type": "Hiding", "traces": "5,000-50,000",
     "overhead": "2-3×", "security": "Medium-High"},
    {"name": "Masking + Hiding", "type": "Combined", "traces": ">1,000,000",
     "overhead": "4-10×", "security": "Very High"},
]

print(f"{'Countermeasure':<20} {'Type':<12} {'Traces Required':<18} {'Overhead':<12} {'Security'}")
print("-" * 85)
for cm in countermeasures:
    print(f"{cm['name']:<20} {cm['type']:<12} {cm['traces']:<18} {cm['overhead']:<12} {cm['security']}")

print("\nKey Takeaways:")
print("1. Masking provides information-theoretic security (provable)")
print("2. Hiding increases attack cost but doesn't eliminate leakage")
print("3. Combined countermeasures provide defense in depth")
print("4. Evaluation should use TVLA + CPA resistance metrics")
print("5. NIST SP 800-90B and FIPS 140-3 require documented evaluation")